# MiniGPT dari Scratch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
import math
import os

## 1. Konfigurasi

In [9]:
VOCAB_SIZE = 50257
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 4
MAX_SEQ_LEN = 512
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 3e-4
GRADIENT_ACCUMULATION  = 4  
SAVE_DIR = 'checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

Device : cuda


## 2. Arsitektur Model

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [11]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(embed_dim, num_heads)
        self.ff    = FeedForward(embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_emb(x)
        pos = self.pos_emb(pos)
        out = self.blocks(tok + pos)
        out = self.norm(out)
        logits = self.head(out)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    def generate(self, x, max_new_tokens, max_seq_len):
        for _ in range(max_new_tokens):
            x_crop = x[:, -max_seq_len:]
            logits, _ = self(x_crop)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)
        return x

In [12]:
model = MiniGPT(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    max_seq_len=MAX_SEQ_LEN
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameter     : {total_params:,}')
print(f'Estimasi ukuran     : {total_params * 4 / 1024 / 1024:.2f} MB')

Total parameter     : 13,774,929
Estimasi ukuran     : 52.55 MB


## 3. Load Dataset

In [13]:
class TextDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.data[idx]['input_ids'], dtype=torch.long)
        return input_ids[:-1], input_ids[1:]


dataset = load_from_disk('data/tokenized')

train_dataset = TextDataset(dataset['train'])
val_dataset = TextDataset(dataset['validation'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train size      : {len(train_dataset)}')
print(f'Validation size : {len(val_dataset)}')

Train size      : 36718
Validation size : 3760


## 4. Training

In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_epoch(epoch):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        loss = loss / GRADIENT_ACCUMULATION
        loss.backward()

        if (batch_idx + 1) % GRADIENT_ACCUMULATION == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch {epoch} | Step {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            total_loss += loss.item()
    avg_loss = total_loss / len(val_loader)
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

In [15]:
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
print(f"Tersimpan di epoch : {checkpoint['epoch']}")
print(f"Val Loss terbaik   : {checkpoint['val_loss']:.4f}")

Tersimpan di epoch : 5
Val Loss terbaik   : 0.7970


C:\Users\User\AppData\Local\Temp\ipykernel_2896\2878489613.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('checkpoints/best_model.pt', map_locat

In [16]:
best_val_loss = float('inf')

checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
print(f"Lanjut dari epoch : {start_epoch}")
print(f"Val Loss terakhir : {checkpoint['val_loss']:.4f}")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f'\n--- Epoch {epoch}/{EPOCHS} ---')
    train_loss = train_epoch(epoch)
    val_loss, perplexity = evaluate()
    scheduler.step()
    print(f'\nEpoch {epoch} Summary')
    print(f'Train Loss  : {train_loss:.4f}')
    print(f'Val Loss    : {val_loss:.4f}')
    print(f'Perplexity  : {perplexity:.2f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, os.path.join(SAVE_DIR, 'best_model.pt'))
        print(f'Model tersimpan di {SAVE_DIR}/best_model.pt')

print('\nTraining selesai!')
print(f'Best Val Loss : {best_val_loss:.4f}')

Lanjut dari epoch : 6
Val Loss terakhir : 0.7970

Training selesai!
Best Val Loss : inf


C:\Users\User\AppData\Local\Temp\ipykernel_2896\91722996.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('checkpoints/best_model.pt', map_locatio

## 5. Evaluasi

In [17]:
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

val_loss, perplexity = evaluate()
print(f'Val Loss   : {val_loss:.4f}')
print(f'Perplexity : {perplexity:.2f}')

C:\Users\User\AppData\Local\Temp\ipykernel_2896\1111883386.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'

Val Loss   : 0.7970
Perplexity : 2.22


In [23]:
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def translate_id_to_en(text):
    return GoogleTranslator(source="id", target="en").translate(text)

def translate_en_to_id(text):
    return GoogleTranslator(source="en", target="id").translate(text)

def generate_indo(prompt_indo, max_new_tokens=10):
    prompt_en = translate_id_to_en(prompt_indo)
    input_ids = tokenizer.encode(prompt_en, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=max_new_tokens, max_seq_len=MAX_SEQ_LEN)
    result_en = tokenizer.decode(output[0], skip_special_tokens=True)
    result_id = translate_en_to_id(result_en)
    return result_id

hasil = generate_indo("apakah kamu ingin bermain dengan ku")
print(hasil)

apakah kamu ingin bermain denganku, komposisi malam Jerman persegi untuk Andy


## 6. Inferensi / Generate Teks

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

prompt = 'you need play with my dog?'

input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

model.eval()
with torch.no_grad():
    output = model.generate(input_ids, max_new_tokens=4, max_seq_len=MAX_SEQ_LEN)

generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f'Prompt  : {prompt}')
print(f'Output  : {generated}')

Prompt  : you need play with my dog?
Output  : you need play with my dog? Yun her review on
